# Predicción temprana de diabetes a partir de variables clínicas
## Proyecto de aula – Minería de Datos – Segundo informe de avance (corte 2)

**Integrantes:** [Nombre completo integrante 1], [Nombre completo integrante 2], [Nombre completo integrante 3]  
**Tipo de proyecto:** Clasificación supervisada (binaria)  
**Fecha de elaboración:** octubre de 2026

---
### 0. Identificación del proyecto

| Elemento | Descripción |
|---|---|
| **Pregunta de investigación** | ¿Es posible predecir tempranamente si una mujer tiene diabetes a partir de variables clínicas básicas (glucosa, IMC, edad, presión arterial, entre otras), y qué modelo de clasificación lo hace mejor, priorizando no dejar pasar casos reales? |
| **Dataset** | *Pima Indians Diabetes Database* (National Institute of Diabetes and Digestive and Kidney Diseases, NIDDK). |
| **Descripción** | 768 registros de mujeres de al menos 21 años de ascendencia indígena Pima (Arizona, EE. UU.), con 8 variables clínicas numéricas y una variable objetivo. |
| **Fuente / enlace** | Repositorio de Jason Brownlee: https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv (original en UCI / Kaggle). |
| **Variable objetivo** | `Outcome`: 1 = tiene diabetes, 0 = no tiene diabetes. |
| **Variables predictoras** | `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`. |

### Cambios respecto al primer corte
1. **Corrección de fuga de datos (*data leakage*):** en el corte 1 los ceros inválidos se imputaron con la mediana calculada sobre *todo* el dataset. Ahora la imputación y el escalado se **ajustan solo con el conjunto de entrenamiento** mediante un `Pipeline` de scikit-learn.
2. El análisis exploratorio ahora está **orientado a preguntas** concretas (no solo gráficos descriptivos).
3. Se agregan **modelos de clasificación y su evaluación**, que no estaban en el corte 1.
4. La pregunta de investigación y el dataset **se mantienen** (se precisó el énfasis en no dejar pasar casos reales).

### 1. Configuración y carga de datos

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

SEED = 42                      # semilla fija para que los resultados sean reproducibles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

columnas = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin',
            'BMI','DiabetesPedigreeFunction','Age','Outcome']
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"

# Se intenta la URL del corte 1; si no hay internet, se usa una copia local del mismo CSV.
try:
    df = pd.read_csv(URL, header=None, names=columnas)
    origen = "URL de GitHub"
except Exception:
    df = pd.read_csv("pima-indians-diabetes.csv", header=None, names=columnas)
    origen = "archivo local pima-indians-diabetes.csv"
print("Datos cargados desde:", origen)
print("Dimensiones (filas, columnas):", df.shape)
df.head()

### 2. Descripción del dataset y revisión de calidad

In [ ]:
print("Tipos de dato:")
print(df.dtypes.to_string())
print("\nValores nulos explícitos (NaN):", int(df.isna().sum().sum()))
print("Filas duplicadas:", int(df.duplicated().sum()))
df.describe().T.round(2)

**Lectura:** el dataset tiene 768 filas y 9 columnas, todas numéricas y sin valores `NaN` explícitos ni filas duplicadas. Sin embargo, `describe()` muestra un **mínimo de 0** en `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin` y `BMI`. Una glucosa, una presión arterial o un IMC de 0 es fisiológicamente imposible: son **valores faltantes disfrazados de cero**. (En `Pregnancies` el 0 sí es válido, y `Outcome` es 0/1 por definición.)

### 3. Preparación de los datos

Se revisan: valores faltantes, duplicados, valores atípicos, fuga de datos, codificación y escalado.

#### 3.1 Valores faltantes (ceros inválidos)

In [ ]:
cols_con_cero_invalido = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

df_na = df.copy()
df_na[cols_con_cero_invalido] = df_na[cols_con_cero_invalido].replace(0, np.nan)

faltantes = pd.DataFrame({
    "n_faltantes": df_na[cols_con_cero_invalido].isna().sum(),
    "porcentaje": (df_na[cols_con_cero_invalido].isna().mean() * 100).round(1)
}).sort_values("porcentaje", ascending=False)
print(faltantes.to_string())

plt.figure(figsize=(7, 3.5))
ax = sns.barplot(x=faltantes.index, y=faltantes["porcentaje"], color="#4C72B0")
ax.bar_label(ax.containers[0], fmt="%.1f%%")
plt.title("Figura 1. Porcentaje de valores faltantes (ceros inválidos) por variable")
plt.xlabel("Variable"); plt.ylabel("% de registros")
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

**Interpretación:** `Insulin` (48.7 %) y `SkinThickness` (29.6 %) tienen muchísimos faltantes; `BloodPressure` (4.6 %), `BMI` (1.4 %) y `Glucose` (0.7 %) tienen pocos. Eliminar las filas con faltantes sacrificaría casi la mitad de los datos, así que se **imputará con la mediana** (robusta a valores extremos). Esa imputación se hará **dentro del `Pipeline`, ajustada solo con entrenamiento**, para evitar fuga de datos (sección 3.4). La alta proporción de faltantes en `Insulin` es una limitación que se retoma en las conclusiones.

#### 3.2 Duplicados

In [ ]:
print("Filas duplicadas:", int(df.duplicated().sum()))

No hay filas duplicadas, por lo que no se elimina ningún registro.

#### 3.3 Valores atípicos (outliers)

In [ ]:
vars_num = columnas[:-1]
filas = []
for c in vars_num:
    s = df_na[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s < lim_inf) | (s > lim_sup)).sum())
    filas.append([c, round(lim_inf, 2), round(lim_sup, 2), n_out, round(100 * n_out / len(s), 1)])
outliers = pd.DataFrame(filas, columns=["variable", "limite_inf", "limite_sup", "n_atipicos", "%"])
print(outliers.to_string(index=False))

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, c in zip(axes.ravel(), vars_num):
    sns.boxplot(y=df_na[c], ax=ax, color="#8FB8DE")
    ax.set_title(c); ax.set_ylabel("")
fig.suptitle("Figura 2. Diagramas de caja por variable (regla del rango intercuartílico)", y=1.02)
plt.tight_layout(); plt.show()

**Interpretación:** hay valores atípicos en casi todas las variables (por ejemplo, `Insulin`, `DiabetesPedigreeFunction` y `Age` tienen colas largas hacia valores altos). Son valores **plausibles clínicamente** (una persona mayor, una insulina muy elevada), no errores de captura. **Decisión:** *no se eliminan*, porque podrían ser justamente los casos de interés y el dataset es pequeño. Se mitiga su efecto usando la **mediana** para imputar y modelos/escalado razonables; se anota como limitación.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, c in zip(axes.ravel(), vars_num):
    sns.histplot(df_na[c].dropna(), bins=25, kde=False, ax=ax, color="#4C72B0")
    ax.set_title(c); ax.set_xlabel(""); ax.set_ylabel("Frecuencia")
fig.suptitle("Figura 2b. Distribución de las variables predictoras (sin los ceros inválidos)", y=1.02)
plt.tight_layout(); plt.show()

**Interpretación:** `BloodPressure`, `SkinThickness` y `BMI` tienen distribuciones aproximadamente simétricas y `Glucose` es ligeramente asimétrica a la derecha; `Pregnancies`, `Age`, `Insulin` y `DiabetesPedigreeFunction` son claramente **asimétricas a la derecha** (muchos valores bajos y pocos muy altos), lo que explica sus valores atípicos y justifica usar la mediana en lugar de la media para imputar.

#### 3.4 Fuga de datos (*data leakage*), codificación y escalado

- **Fuga de datos:** si se calcula la mediana (o la media/desviación para escalar) con *todo* el dataset, el modelo recibe información del conjunto de prueba, y las métricas salen algo optimistas. Por eso: (1) primero se **separa** entrenamiento/prueba (80 %/20 %, estratificado por `Outcome`) y (2) la imputación y el escalado viven dentro de un `Pipeline`, de modo que se **ajustan solo con entrenamiento** y luego se aplican a prueba. Además, ninguna variable predictora es una consecuencia directa del diagnóstico (no hay columnas derivadas de `Outcome`).
- **Identificadores y variables innecesarias:** el dataset no incluye columnas identificadoras (ID, nombre, documento) ni variables derivadas del diagnóstico, por lo que **se conservan las ocho predictoras**; tampoco hay categorías inconsistentes porque no hay variables de texto.
- **Codificación:** todas las variables son numéricas y `Outcome` ya es 0/1; **no se requiere codificación** de categorías.
- **Escalado:** la regresión logística es sensible a la escala (usa `StandardScaler`); el árbol de decisión no lo necesita, por lo que no se escala.

In [ ]:
X = df_na.drop(columns="Outcome")      # predictoras (con NaN en lugar de ceros inválidos)
y = df_na["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

resumen_split = pd.DataFrame({
    "filas": [len(y_train), len(y_test)],
    "% diabetes (Outcome=1)": [round(100 * y_train.mean(), 1), round(100 * y_test.mean(), 1)]
}, index=["Entrenamiento", "Prueba"])
resumen_split

**Interpretación:** la separación estratificada deja ~35 % de casos con diabetes tanto en entrenamiento (614 filas) como en prueba (154 filas), así que ambos conjuntos son comparables.

### 4. Análisis exploratorio de datos (EDA) orientado a preguntas

> **Aviso importante:** el EDA describe **asociaciones**, no causas. Que dos variables estén relacionadas no significa que una cause la otra. Aquí solo se describe (sin imputar ni entrenar nada), por lo que no genera fuga de datos.

**Preguntas guía:**
- **P1.** ¿Qué tan desbalanceada está la variable objetivo?
- **P2.** ¿Qué variables clínicas se distinguen más entre personas con y sin diabetes?
- **P3.** ¿Qué variables están más asociadas con `Outcome` y entre sí?
- **P4.** ¿Cómo cambia la proporción de diabetes según la edad y el IMC?

#### P1. Balance de la variable objetivo

In [ ]:
conteo = df["Outcome"].value_counts().sort_index()
porc = (df["Outcome"].value_counts(normalize=True).sort_index() * 100).round(1)
print(pd.DataFrame({"casos": conteo, "porcentaje": porc}).rename(index={0: "Sin diabetes (0)", 1: "Con diabetes (1)"}).to_string())

plt.figure(figsize=(5, 3.8))
ax = sns.countplot(x="Outcome", data=df, palette=["#4C72B0", "#C44E52"])
ax.bar_label(ax.containers[0])
ax.set_xticklabels(["Sin diabetes (0)", "Con diabetes (1)"])
plt.title("Figura 3. Distribución de la variable objetivo")
plt.xlabel("Diagnóstico"); plt.ylabel("Número de mujeres")
plt.tight_layout(); plt.show()

**Interpretación:** 500 mujeres (65.1 %) sin diabetes y 268 (34.9 %) con diabetes: hay un **desbalance moderado**. Consecuencia práctica: un modelo que siempre responda "sin diabetes" obtendría ~65 % de *accuracy* sin detectar a nadie. Por eso la exactitud sola no basta y se evaluarán precisión, *recall* y F1 (sección 6).

#### P2. ¿Qué variables difieren entre personas con y sin diabetes?

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, c in zip(axes.ravel(), vars_num):
    sns.boxplot(x="Outcome", y=c, data=df_na, ax=ax, palette=["#8FB8DE", "#E7969A"])
    ax.set_xticklabels(["Sin diab.", "Con diab."]); ax.set_xlabel("")
    ax.set_title(c)
fig.suptitle("Figura 4. Distribución de cada variable según el diagnóstico", y=1.02)
plt.tight_layout(); plt.show()

medianas = df_na.groupby("Outcome")[vars_num].median().T
medianas.columns = ["Mediana sin diabetes", "Mediana con diabetes"]
medianas["Diferencia"] = (medianas.iloc[:, 1] - medianas.iloc[:, 0]).round(2)
print("Tabla 1. Medianas por grupo")
medianas.round(2)

**Interpretación:** la diferencia más marcada es la de **`Glucose`** (mediana 140 con diabetes vs. 107 sin diabetes). Le siguen `BMI` (34.3 vs. 30.1), `Age` (36 vs. 27 años) e `Insulin` (169.5 vs. 102.5, pero con casi la mitad de datos faltantes). `BloodPressure` casi no se diferencia (74.5 vs. 70). Es decir: las mujeres con diabetes tienden a presentar glucosa, IMC y edad más altos. Es una **asociación** observada en este dataset, no una relación causal.

#### P3. Asociaciones entre variables (correlación)

In [ ]:
corr = df_na[columnas].corr()          # usa los datos disponibles (pares sin NaN)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Figura 5. Matriz de correlación de Pearson")
plt.tight_layout(); plt.show()

print("Tabla 2. Correlación de cada variable con Outcome (de mayor a menor)")
corr["Outcome"].drop("Outcome").sort_values(ascending=False).round(3).to_frame("correlación")

**Interpretación:** `Glucose` es la variable más asociada con `Outcome` (r = 0.50), seguida de `BMI` (0.31), `Insulin` (0.30), `SkinThickness` (0.26), `Age` (0.24) y `Pregnancies` (0.22). Todas las correlaciones con `Outcome` son positivas pero de **intensidad baja a moderada**, por lo que ninguna variable aislada separa perfectamente los grupos; el modelo deberá combinarlas. Entre predictoras también hay asociaciones esperables (p. ej. edad–embarazos), lo que puede restar interpretabilidad a los coeficientes de la regresión logística.

#### P4. Proporción de diabetes por grupos de edad y de IMC

In [ ]:
d = df_na.copy()
d["grupo_edad"] = pd.cut(d["Age"], [20, 30, 40, 50, 90], right=False,
                         labels=["21-29", "30-39", "40-49", "50+"])
d["grupo_imc"] = pd.cut(d["BMI"], [0, 18.5, 25, 30, 100], right=False,
                        labels=["Bajo peso (<18.5)", "Normal (18.5-24.9)", "Sobrepeso (25-29.9)", "Obesidad (≥30)"])

tabla_edad = d.groupby("grupo_edad", observed=True)["Outcome"].agg(["count", "mean"])
tabla_imc = d.groupby("grupo_imc", observed=True)["Outcome"].agg(["count", "mean"])
tabla_edad["mean"] = (tabla_edad["mean"] * 100).round(1)
tabla_imc["mean"] = (tabla_imc["mean"] * 100).round(1)
print("Tabla 3. % de diabetes por grupo de edad"); print(tabla_edad.rename(columns={"count": "n", "mean": "% diabetes"}).to_string())
print("\nTabla 4. % de diabetes por categoría de IMC"); print(tabla_imc.rename(columns={"count": "n", "mean": "% diabetes"}).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sns.barplot(x=tabla_edad.index.astype(str), y=tabla_edad["mean"], ax=axes[0], color="#4C72B0")
axes[0].bar_label(axes[0].containers[0], fmt="%.1f%%")
axes[0].set_title("Figura 6a. % con diabetes por grupo de edad"); axes[0].set_xlabel("Edad (años)"); axes[0].set_ylabel("% con diabetes")
sns.barplot(x=tabla_imc.index.astype(str), y=tabla_imc["mean"], ax=axes[1], color="#C44E52")
axes[1].bar_label(axes[1].containers[0], fmt="%.1f%%")
axes[1].set_title("Figura 6b. % con diabetes por categoría de IMC"); axes[1].set_xlabel("Categoría de IMC"); axes[1].set_ylabel("% con diabetes")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

**Interpretación:** la proporción de diabetes pasa de **21.2 %** en mujeres de 21–29 años a **46.1 %** en 30–39 y **55.1 %** en 40–49; en el grupo de 50+ es 48.3 % (no sigue subiendo, y tiene solo 89 casos). Con el IMC la tendencia es clara: **6.9 %** con IMC normal, **22.3 %** con sobrepeso y **46.4 %** con obesidad. (El grupo "bajo peso" tiene solo 4 casos, por lo que no se interpreta.) De nuevo son asociaciones descriptivas; no se puede afirmar que la edad o el IMC *causen* la diabetes a partir de estos datos.

#### Síntesis del EDA
1. Hay desbalance moderado (34.9 % con diabetes): no basta con la *accuracy*.
2. `Glucose` es la variable más asociada con el diagnóstico; `BMI`, `Age` e `Insulin` también aportan.
3. La proporción de diabetes aumenta con la edad (hasta los 40-49) y con el IMC.
4. Hay muchos faltantes en `Insulin` y `SkinThickness`, lo que limita esas variables.

### 5. Modelado

Se comparan **dos familias de modelos** de clasificación, con la **misma partición** entrenamiento/prueba y la misma semilla:

1. **Regresión logística** (modelo lineal, interpretable; con escalado).
2. **Árbol de decisión** (modelo basado en reglas; con `max_depth=4` y `min_samples_leaf=10` fijos para evitar sobreajuste; *no se hizo búsqueda de hiperparámetros*).

Para cada uno se prueba también la variante con `class_weight="balanced"`, que penaliza más los errores sobre la clase minoritaria (diabetes), pensando en reducir los falsos negativos. Se incluye además un **modelo de referencia** (*baseline*) que siempre predice la clase mayoritaria, para saber si los modelos aportan algo real.

In [ ]:
def construir(modelo, escalar):
    pasos = [("imputador", SimpleImputer(strategy="median"))]   # se ajusta solo con entrenamiento
    if escalar:
        pasos.append(("escalador", StandardScaler()))
    pasos.append(("modelo", modelo))
    return Pipeline(pasos)

modelos = {
    "Baseline (clase mayoritaria)": construir(DummyClassifier(strategy="most_frequent"), False),
    "Regresión logística": construir(LogisticRegression(max_iter=1000, random_state=SEED), True),
    "Regresión logística (balanceada)": construir(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED), True),
    "Árbol de decisión": construir(DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=SEED), False),
    "Árbol de decisión (balanceado)": construir(DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, class_weight="balanced", random_state=SEED), False),
}

predicciones, probabilidades = {}, {}
for nombre, pipe in modelos.items():
    pipe.fit(X_train, y_train)
    predicciones[nombre] = pipe.predict(X_test)
    probabilidades[nombre] = pipe.predict_proba(X_test)[:, 1]
print("Modelos entrenados:", len(modelos))

### 6. Evaluación

Se calculan las métricas sobre el **conjunto de prueba** (datos que el modelo no vio). La clase positiva es **diabetes (1)**:
- *Accuracy*: proporción total de aciertos.
- *Precisión*: de los que el modelo dice "diabetes", cuántos realmente lo son (cuida los **falsos positivos**).
- *Recall* (sensibilidad): de los que realmente tienen diabetes, cuántos detecta el modelo (cuida los **falsos negativos**).
- *F1*: balance entre precisión y recall.

In [ ]:
filas = []
for nombre, pred in predicciones.items():
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    filas.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, pred),
        "Precisión": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, probabilidades[nombre]),
        "VP": tp, "FP": fp, "FN": fn, "VN": tn,
    })
resultados = pd.DataFrame(filas).set_index("Modelo")
print("Tabla 5. Métricas en el conjunto de prueba (clase positiva = diabetes)")
resultados.round(3)

#### 6.1 Resultados por clase

In [ ]:
for nombre in ["Regresión logística", "Árbol de decisión",
               "Regresión logística (balanceada)", "Árbol de decisión (balanceado)"]:
    print("=" * 62)
    print(nombre)
    print(classification_report(y_test, predicciones[nombre],
                                target_names=["Sin diabetes (0)", "Con diabetes (1)"], digits=3))

#### 6.2 Matrices de confusión

In [ ]:
orden = ["Regresión logística", "Regresión logística (balanceada)",
         "Árbol de decisión", "Árbol de decisión (balanceado)"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))
for ax, nombre in zip(axes, orden):
    ConfusionMatrixDisplay.from_predictions(
        y_test, predicciones[nombre], display_labels=["Sin diab.", "Con diab."],
        cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Valor real")
fig.suptitle("Figura 7. Matrices de confusión en el conjunto de prueba (n = 154)", y=1.03)
plt.tight_layout(); plt.show()

#### 6.3 Estabilidad: validación cruzada en entrenamiento

Con solo 154 casos de prueba (54 con diabetes), una única partición puede ser engañosa: un cambio de pocos casos mueve bastante las métricas. Como verificación adicional se usa **validación cruzada estratificada de 5 pliegues sobre el conjunto de entrenamiento** (no toca el de prueba; no es ajuste de hiperparámetros).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
filas = []
for nombre, pipe in modelos.items():
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["accuracy", "precision", "recall", "f1"])
    filas.append({"Modelo": nombre,
                  "Accuracy": r["test_accuracy"].mean(), "Precisión": r["test_precision"].mean(),
                  "Recall": r["test_recall"].mean(), "Recall (desv.)": r["test_recall"].std(),
                  "F1": r["test_f1"].mean()})
cv_res = pd.DataFrame(filas).set_index("Modelo")
print("Tabla 6. Validación cruzada (5 pliegues) en entrenamiento")
cv_res.round(3)

#### 6.4 Comparación visual

In [ ]:
metricas = ["Accuracy", "Precisión", "Recall", "F1"]
ax = resultados.drop(index="Baseline (clase mayoritaria)")[metricas].plot(
    kind="bar", figsize=(11, 4.5), width=0.8, colormap="tab10")
plt.title("Figura 8. Comparación de métricas por modelo (conjunto de prueba)")
plt.ylabel("Valor"); plt.xlabel(""); plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right"); plt.legend(loc="lower right", ncol=4)
plt.tight_layout(); plt.show()

#### 6.5 ¿Qué variables usan los modelos?

In [ ]:
coef = modelos["Regresión logística (balanceada)"].named_steps["modelo"].coef_[0]
imp = modelos["Árbol de decisión (balanceado)"].named_steps["modelo"].feature_importances_
tabla_vars = pd.DataFrame({"Coef. reg. logística (estandarizado)": coef,
                           "Importancia árbol": imp}, index=X.columns).sort_values("Importancia árbol", ascending=False)
print("Tabla 7. Peso de cada variable en los modelos balanceados")
tabla_vars.round(3)

**Interpretación:** en ambos modelos `Glucose` es la variable dominante, coherente con el EDA (era la más asociada con `Outcome`). Los coeficientes se leen como asociación con la probabilidad de diabetes, no como causalidad.

#### 6.6 Selección del modelo y lectura de los errores

**Criterio:** no se elige solo por *accuracy*. En un tamizaje temprano, un **falso negativo** (decirle a una mujer con diabetes que "no tiene") es más grave que un **falso positivo** (enviarla a un examen confirmatorio que resulta normal). Por eso se prioriza el *recall* de la clase diabetes, cuidando que la precisión y el F1 no se desplomen, y que el resultado sea **consistente** entre la prueba y la validación cruzada.

**Lectura de las Tablas 5 y 6:**
- El *baseline* logra 64.9 % de *accuracy* con **recall = 0**: la exactitud sola engaña. Todos los modelos lo superan en detección.
- **Regresión logística estándar:** *accuracy* de 70.8 % en prueba, pero **recall de 0.50** (27 falsos negativos de 54 casos reales): deja pasar la mitad de los casos.
- **Árbol de decisión estándar** tiene la mejor *accuracy* (79.2 %) y F1 (0.709) en prueba, pero en validación cruzada baja a *accuracy* 72.5 % y recall 0.542: el resultado en prueba parece, en parte, **suerte de la partición**.
- **Regresión logística balanceada:** recall de 0.704 y F1 de 0.650 en prueba (16 falsos negativos); en validación cruzada recall 0.701 y F1 0.672. **Es el modelo más consistente entre ambas evaluaciones.**
- **Árbol balanceado:** el mayor recall en prueba (0.852; solo 8 falsos negativos), pero con 36 falsos positivos (precisión 0.561) y *accuracy* de 71.4 %; en validación cruzada su recall es 0.734.

**Decisión:** el mejor modelo **para este problema** es la **regresión logística balanceada**: detecta ~70 % de los casos con un recall estable entre evaluaciones, F1 competitivo, es interpretable y no depende tanto de una partición favorable. Si el criterio fuera estrictamente minimizar falsos negativos sin importar los falsos positivos, el árbol balanceado sería la alternativa (a costa de más falsas alarmas). Ninguno de los modelos es apto para uso clínico real.

### 7. Conclusiones

**1. Hallazgos principales del EDA.** (a) El dataset tiene 34.9 % de casos con diabetes (desbalance moderado). (b) Cinco variables traen ceros inválidos; `Insulin` (48.7 %) y `SkinThickness` (29.6 %) son las más afectadas. (c) `Glucose` es la variable más asociada con el diagnóstico (r = 0.50; mediana 140 vs. 107). (d) La proporción de diabetes crece con el IMC (6.9 % en IMC normal → 46.4 % en obesidad) y con la edad (21.2 % en 21–29 años → 55.1 % en 40–49). Todo esto son **asociaciones**, no causas.

**2. Mejor modelo.** La **regresión logística con `class_weight="balanced"`**, frente al árbol de decisión, la regresión logística estándar y el *baseline*.

**3. Evidencia.** En prueba: recall 0.704, precisión 0.603, F1 0.650, accuracy 0.734 y 16 falsos negativos; en validación cruzada (5 pliegues): recall 0.701 y F1 0.672. Es el modelo cuyo desempeño se mantiene más parecido entre ambas evaluaciones; el árbol estándar parecía mejor en *accuracy* de prueba (0.792) pero cae en validación cruzada (0.725).

**4. Errores y limitaciones.**
- El modelo elegido **aún deja pasar 16 de 54 casos reales (≈30 %)** y genera 25 falsos positivos.
- El conjunto de prueba es pequeño (154 filas), por lo que las métricas tienen **incertidumbre alta** (una partición distinta puede cambiar los resultados).
- Muchos faltantes en `Insulin` y `SkinThickness` imputados con la mediana, lo cual puede distorsionar esas variables.
- Los datos son de **mujeres Pima ≥ 21 años**: los resultados no se pueden generalizar a otras poblaciones.
- No se ajustaron hiperparámetros ni se probaron otros modelos; los valores atípicos se conservaron.
- Las relaciones encontradas son asociaciones; no se puede afirmar causalidad.

**Trabajo pendiente para el corte final:** ajuste de hiperparámetros, otros modelos (p. ej. bosque aleatorio), ajuste del umbral de decisión para subir el recall y mejor tratamiento de faltantes.

### 8. Bitácora de uso de inteligencia artificial

> Esta sección debe ser **completada por los integrantes** con los usos reales de IA (herramienta, propósito, *prompt* completo, problemas identificados, ajustes realizados y decisión final), tal como lo exige la guía del avance. El detalle completo se presenta en el informe PDF.

| # | Herramienta | Propósito | Prompt utilizado | Problemas identificados | Ajustes realizados | Decisión final |
|---|---|---|---|---|---|---|
| 1 | Claude (Anthropic) | Apoyo en la construcción del cuaderno del corte 2 | [pegar el prompt real] | [completar] | [completar] | [completar] |

**Declaración de responsabilidad:** [pegar aquí el texto textual de la guía].

---
## Tercer corte – Revisión, optimización y preparación del modelo para Streamlit

**Actividad 1 – Integración y despliegue del modelo.** Esta sección **se agrega al final** del notebook del corte 2 (no se repite el EDA). Usa las variables ya creadas arriba (`X_train`, `X_test`, `y_train`, `y_test`, `cv`, `resultados`, `modelos`).

### 9. Revisión del rendimiento del modelo del corte 2

Modelo del corte 2: **regresión logística balanceada** (accuracy 0.734, recall 0.704, **F1 0.650**, 16 falsos negativos). Aunque el accuracy supera 0.70, el **F1 (0.65) y la precisión (0.60) están por debajo de 0.70**, así que se aplica la *Ficha técnica de revisión de calidad del modelado* antes de integrarlo:

| Punto de revisión | Estado en este proyecto |
|---|---|
| Tratamiento de los datos | Ceros inválidos → `NaN` en 5 variables; imputación por mediana **dentro del `Pipeline`** |
| Separación entrenamiento/prueba | 80/20 estratificada, semilla 42; la prueba **no se usa** para decidir nada |
| Fuga de información | Imputación y escalado ajustados solo con entrenamiento; ninguna predictora deriva de `Outcome` |
| Desbalance | 34.9 % de casos positivos → `class_weight="balanced"` y métricas por clase |
| Colinealidad | Revisada en la matriz de correlación (edad–embarazos); sin variables redundantes graves |
| Sobreajuste e hiperparámetros | **Pendiente en el corte 2** (árbol y regresión con parámetros fijos) → se corrige aquí |
| Estabilidad | Validación cruzada de 5 pliegues, con el *recall* por clase como criterio |

**Cambios que se hacen:** (1) búsqueda de hiperparámetros con validación cruzada para regresión logística y bosque aleatorio, con una métrica que prioriza detectar casos reales (**F2**, que pesa más el *recall*); (2) ajuste del **umbral de decisión** usando solo predicciones *out-of-fold* de entrenamiento; (3) comparación *antes vs. después* en el conjunto de prueba.

In [ ]:
import json
import joblib
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import make_scorer, fbeta_score

DATASET_PATH = "pima-indians-diabetes.csv"      # mismo nombre que usa la app de Streamlit
os.makedirs("models", exist_ok=True)
modelo_corte2 = modelos["Regresión logística (balanceada)"]   # modelo del corte 2 (referencia "antes")

#### 9.1 Búsqueda de hiperparámetros (solo con entrenamiento)

In [ ]:
f2 = make_scorer(fbeta_score, beta=2)
scoring = {"f2": f2, "recall": "recall", "precision": "precision", "f1": "f1", "roc_auc": "roc_auc"}

candidatos = {
    "Regresión logística (balanceada, ajustada)": (
        Pipeline([("imputador", SimpleImputer(strategy="median")), ("escalador", StandardScaler()),
                  ("modelo", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))]),
        {"modelo__C": [0.01, 0.05, 0.1, 0.5, 1, 5, 10]}),
    "Bosque aleatorio (balanceado)": (
        Pipeline([("imputador", SimpleImputer(strategy="median")),
                  ("modelo", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                    random_state=SEED))]),
        {"modelo__max_depth": [3, 4, 6, 8], "modelo__min_samples_leaf": [5, 10, 20]}),
}

busquedas, filas = {}, []
for nombre, (pipe, grid) in candidatos.items():
    gs = GridSearchCV(pipe, grid, cv=cv, scoring=scoring, refit="f2", n_jobs=-1)
    gs.fit(X_train, y_train)
    busquedas[nombre] = gs
    i = gs.best_index_
    filas.append({"Modelo": nombre, "Mejores parámetros": gs.best_params_,
                  "F2 (CV)": gs.cv_results_["mean_test_f2"][i],
                  "Recall (CV)": gs.cv_results_["mean_test_recall"][i],
                  "Precisión (CV)": gs.cv_results_["mean_test_precision"][i],
                  "F1 (CV)": gs.cv_results_["mean_test_f1"][i],
                  "ROC-AUC (CV)": gs.cv_results_["mean_test_roc_auc"][i]})
tabla_busqueda = pd.DataFrame(filas).set_index("Modelo")
print("Tabla 8. Mejor configuración de cada modelo (validación cruzada de 5 pliegues, solo entrenamiento)")
tabla_busqueda.round(3)

#### 9.2 Selección del modelo y ajuste del umbral de decisión

In [ ]:
# Modelo con mayor F2 en validación cruzada
nombre_final = max(busquedas, key=lambda n: busquedas[n].best_score_)
modelo_final = busquedas[nombre_final].best_estimator_
print("Modelo seleccionado:", nombre_final, busquedas[nombre_final].best_params_)

# Umbral: se elige con probabilidades out-of-fold del ENTRENAMIENTO (la prueba no interviene)
proba_oof = cross_val_predict(modelo_final, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
umbrales = np.round(np.arange(0.20, 0.81, 0.01), 2)
filas = []
for t in umbrales:
    p = (proba_oof >= t).astype(int)
    filas.append({"umbral": t, "F2": fbeta_score(y_train, p, beta=2),
                  "recall": recall_score(y_train, p), "precision": precision_score(y_train, p, zero_division=0)})
curva_umbral = pd.DataFrame(filas).set_index("umbral")
UMBRAL = float(curva_umbral["F2"].idxmax())
print(f"Umbral elegido (máximo F2 en entrenamiento, out-of-fold): {UMBRAL}")

ax = curva_umbral.plot(figsize=(7, 3.8))
ax.axvline(UMBRAL, color="gray", linestyle="--")
plt.title("Figura 9. Métricas según el umbral de decisión (validación cruzada en entrenamiento)")
plt.xlabel("Umbral de probabilidad"); plt.tight_layout(); plt.show()

#### 9.3 Evaluación final en prueba: antes vs. después

In [ ]:
def evaluar(y_true, proba, umbral):
    pred = (proba >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {"Accuracy": accuracy_score(y_true, pred), "Precisión": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred), "F1": f1_score(y_true, pred),
            "F2": fbeta_score(y_true, pred, beta=2), "ROC-AUC": roc_auc_score(y_true, proba),
            "VP": int(tp), "FP": int(fp), "FN": int(fn), "VN": int(tn)}

proba_antes = modelo_corte2.predict_proba(X_test)[:, 1]
proba_final = modelo_final.predict_proba(X_test)[:, 1]
comparacion = pd.DataFrame({
    "Corte 2: reg. logística balanceada (umbral 0.5)": evaluar(y_test, proba_antes, 0.5),
    f"Corte 3: {nombre_final} (umbral 0.5)": evaluar(y_test, proba_final, 0.5),
    f"Corte 3: {nombre_final} (umbral {UMBRAL})": evaluar(y_test, proba_final, UMBRAL),
}).T
print("Tabla 9. Métricas en el conjunto de prueba (clase positiva = diabetes)")
comparacion.round(3)

In [ ]:
# Alerta de revisión: ¿alguna métrica principal por debajo de 0.70?
final = comparacion.iloc[-1]
for m in ["Accuracy", "Recall", "F1"]:
    estado = "OK" if final[m] >= 0.70 else "REVISAR (< 0.70)"
    print(f"{m:10s} {final[m]:.3f}  {estado}")
print(f"Falsos negativos: {int(final['FN'])} de {int(final['FN'] + final['VP'])} casos reales")

pred_final = (proba_final >= UMBRAL).astype(int)
ConfusionMatrixDisplay.from_predictions(y_test, pred_final, display_labels=["Sin diab.", "Con diab."],
                                        cmap="Blues", colorbar=False)
plt.title(f"Figura 10. Matriz de confusión final (umbral {UMBRAL}, n = {len(y_test)})")
plt.tight_layout(); plt.show()

In [ ]:
# Revisión de la Ficha técnica: recall por clase, sobreajuste, estabilidad y colinealidad
rec_pos = recall_score(y_test, pred_final)                  # sensibilidad (con diabetes)
rec_neg = recall_score(y_test, pred_final, pos_label=0)     # especificidad (sin diabetes)

pred_train = (modelo_final.predict_proba(X_train)[:, 1] >= UMBRAL).astype(int)
f1_train, f1_test = f1_score(y_train, pred_train), f1_score(y_test, pred_final)

f1_folds, rec_folds = [], []                                # estabilidad por pliegue (umbral final)
for _, idx in cv.split(X_train, y_train):
    p_ = (proba_oof[idx] >= UMBRAL).astype(int)
    f1_folds.append(f1_score(y_train.iloc[idx], p_)); rec_folds.append(recall_score(y_train.iloc[idx], p_))

c_abs = X.corr().abs().where(~np.eye(X.shape[1], dtype=bool))
par = c_abs.stack().idxmax(); corr_max = float(c_abs.stack().max())

revision = {
    "recall_con_diabetes": round(rec_pos, 4), "recall_sin_diabetes": round(rec_neg, 4),
    "f1_entrenamiento": round(f1_train, 4), "f1_prueba": round(f1_test, 4),
    "brecha_f1": round(f1_train - f1_test, 4),
    "cv_recall_media": round(float(np.mean(rec_folds)), 4), "cv_recall_desv": round(float(np.std(rec_folds)), 4),
    "cv_f1_media": round(float(np.mean(f1_folds)), 4), "cv_f1_desv": round(float(np.std(f1_folds)), 4),
    "corr_max": round(corr_max, 3), "corr_max_par": f"{par[0]} - {par[1]}",
}
for k_, v_ in revision.items(): print(f"{k_:22s} {v_}")
print("\nSobreajuste:", "posible (brecha F1 > 0.10)" if revision["brecha_f1"] > 0.10 else "no evidente (brecha F1 <= 0.10)")
print("Estabilidad: desv. del recall entre pliegues =", revision["cv_recall_desv"])
print("Colinealidad:", "alta (> 0.80)" if corr_max > 0.80 else "no problemática (máx |r| <= 0.80)")

#### 9.4 Revisión por alerta (< 0.70), interpretación y decisión final

> **Completar con los resultados reales de las Tablas 8 y 9 y la Figura 10** (los números dependen de la ejecución): ¿mejoró el *recall* y el F1 frente al corte 2? ¿cuántos falsos negativos y falsos positivos hay ahora? ¿el modelo se mantiene estable entre validación cruzada y prueba? Si el F1 o el accuracy siguen por debajo de 0.70, dejarlo escrito como limitación (el 70 % es una señal de revisión, no una garantía).

**Qué mirar en la revisión (Ficha técnica):** *recall* por clase y falsos negativos; brecha de F1 entre entrenamiento y prueba (sobreajuste); desviación del *recall* entre pliegues (estabilidad); y la correlación máxima entre predictoras (colinealidad).

**Regla de lectura:** el umbral más bajo detecta más casos reales (menos falsos negativos) a costa de más falsas alarmas; en un tamizaje temprano eso es preferible, siempre con confirmación clínica.

### 10. Guardado del pipeline completo para la app de Streamlit

In [ ]:
# El pipeline incluye imputación, escalado (si aplica) y modelo: la app NO vuelve a entrenar.
joblib.dump(modelo_final, "models/pipeline_diabetes.joblib")

def a_float(d): return {k: (int(v) if k in ("VP", "FP", "FN", "VN") else round(float(v), 4)) for k, v in d.items()}
cv_final = busquedas[nombre_final]; i = cv_final.best_index_
metricas = {
    "modelo_final": nombre_final,
    "hiperparametros": {k: (v.item() if hasattr(v, "item") else v) for k, v in cv_final.best_params_.items()},
    "umbral": UMBRAL,
    "sklearn_version": sklearn.__version__,
    "n_total": int(len(df)), "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "pct_diabetes": round(float(y.mean() * 100), 1),
    "test_final": a_float(comparacion.iloc[-1].to_dict()),
    "test_corte2": a_float(comparacion.iloc[0].to_dict()),
    "cv_final": {"F2": round(float(cv_final.cv_results_["mean_test_f2"][i]), 4),
                 "Recall": round(float(cv_final.cv_results_["mean_test_recall"][i]), 4),
                 "Precisión": round(float(cv_final.cv_results_["mean_test_precision"][i]), 4),
                 "F1": round(float(cv_final.cv_results_["mean_test_f1"][i]), 4),
                 "ROC-AUC": round(float(cv_final.cv_results_["mean_test_roc_auc"][i]), 4)},
    "revision": revision,
    "comparacion_test": {k: a_float(v) for k, v in comparacion.to_dict("index").items()},
}
with open("models/metricas.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)

# Registros de prueba con la predicción del notebook, para verificar la app (misma entrada → misma salida)
prueba = X_test.copy()
prueba["Outcome_real"] = y_test
prueba["Probabilidad_notebook"] = proba_final.round(6)
prueba["Prediccion_notebook"] = pred_final
prueba.index.name = "fila_dataset"
prueba.to_csv("models/registros_prueba.csv")
print("Guardados: models/pipeline_diabetes.joblib, models/metricas.json, models/registros_prueba.csv")
print("Versión de scikit-learn usada:", sklearn.__version__, "(debe coincidir con requirements.txt)")

#### 10.1 Verificación: recargar el pipeline y predecir un registro conocido

In [ ]:
pipe_cargado = joblib.load("models/pipeline_diabetes.joblib")
registro = X_test.iloc[[0]]                              # un registro del conjunto de prueba
p = pipe_cargado.predict_proba(registro)[0, 1]
print("Probabilidad de diabetes (pipeline recargado):", round(p, 6))
print("Probabilidad en el notebook:                  ", round(proba_final[0], 6))
assert abs(p - proba_final[0]) < 1e-9
print("Predicción:", "diabetes" if p >= UMBRAL else "sin diabetes", f"(umbral {UMBRAL})")